# Publication figure and table consistency repairs

This notebook regenerates Figure 4, Figures F1–F4, and Table H2 from saved participant-level predictions and repeated-validation summaries. It does not select new models. The matched-$k$ differential figures are generated by the corrected Notebook 07.

## tl;dr

- Figure 4 now compares PSD, HCFIM, and PSD+HCFIM using the same Extra Trees model, repeated 10×5 cross-validation, and $k=30$.
- Figures F1–F4 retain the original exploratory models but report integer confusion-matrix counts and sample sizes.
- Table H2 assigns inner-selection frequencies only to the adaptive rows and explicitly flags the below-chance fixed-$m=3$ AD result.
- All outputs are regenerated from the project CSV files listed below.

## Context and methods

### Key assumptions

- Percentile bars in Figure 4 are 2.5th–97.5th percentiles across repetitions, not confidence intervals.
- F1–F4 remain descriptive records of the original nested partitions and are not promoted over repeated validation.
- No probability inversion or post hoc threshold optimization is performed.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT = resolve_project_root()
RESULTS = PROJECT / "results"
FIGURES = PROJECT / "figures"
REPAIR_RESULTS = RESULTS / "publication_consistency_repairs"
REPAIR_RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
COLORS = {"PSD": "#72a1c5", "HCFIM": "#51beb6", "PSD+HCFIM": "#005a9f"}
print(PROJECT)

## Results

### 1. Regenerate Figure 4 with matched model capacity and feature count

In [ ]:
summary_path = RESULTS / "repeated_primary_validation" / "repeated_primary_summary.csv"
summary = pd.read_csv(summary_path)
figure4 = summary.loc[
    (summary["k"] == 30)
    & summary["feature_set"].isin(["PSD", "HCFIM", "PSD+HCFIM"])
].copy()
task_labels = {
    "control_vs_alzheimer": "Control vs AD",
    "control_vs_frontotemporal_dementia": "Control vs FTD",
}
figure4["comparison"] = figure4["task"].map(task_labels)
feature_order = ["PSD", "HCFIM", "PSD+HCFIM"]
comparison_order = ["Control vs AD", "Control vs FTD"]
assert len(figure4) == 6

fig, ax = plt.subplots(figsize=(9.2, 4.9))
x = np.arange(len(comparison_order))
width = 0.23
for offset_index, feature_set in enumerate(feature_order):
    rows = figure4.set_index(["comparison", "feature_set"]).loc[
        [(comparison, feature_set) for comparison in comparison_order]
    ].reset_index()
    xpos = x + (offset_index - 1) * width
    means = rows["auc_mean"].to_numpy()
    low = means - rows["auc_p2_5"].to_numpy()
    high = rows["auc_p97_5"].to_numpy() - means
    bars = ax.bar(xpos, means, width, color=COLORS[feature_set], label=feature_set)
    ax.errorbar(xpos, means, yerr=np.vstack([low, high]), fmt="none", ecolor="#222222", capsize=4, linewidth=1.2)
    for bar, value in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, value + 0.012, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

ax.axhline(0.5, color="#666666", linestyle="--", linewidth=1, label="Chance")
ax.set_xticks(x, comparison_order)
ax.set_ylabel("Mean ROC AUC")
ax.set_ylim(0.45, 0.88)
#ax.set_title("Repeated participant-level validation (10×5 CV, Extra Trees, k=30)")
ax.legend(frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.01))
sns.despine(ax=ax)
fig.tight_layout()
figure4_path = FIGURES / "figure_4_repeated_matched_k_auc.png"
fig.savefig(figure4_path, dpi=300, bbox_inches="tight")
plt.show()

figure4[["comparison", "feature_set", "k", "auc_mean", "auc_p2_5", "auc_p97_5"]].to_csv(
    REPAIR_RESULTS / "figure_4_values.csv", index=False
)
display(figure4[["comparison", "feature_set", "auc_mean", "auc_p2_5", "auc_p97_5"]])

### 2. Regenerate Figures F1–F4 with integer matrix annotations

In [ ]:
figure_specs = [
    {
        "label": "F1", "file": "oof_predictions_control_vs_alzheimer_psd_plus_hcfim_extra_trees.csv",
        "output": "classification_best_control_vs_alzheimer.png", "positive": "AD",
    },
    {
        "label": "F2", "file": "oof_predictions_control_vs_frontotemporal_dementia_hcfim_mlp_small.csv",
        "output": "classification_best_control_vs_frontotemporal_dementia.png", "positive": "FTD",
    },
    {
        "label": "F3", "file": "oof_predictions_lateralized_v1_control_vs_alzheimer_psd_plus_hcfim_plus_lateral_deltas_extra_trees.csv",
        "output": "classification_best_lateralized_v1_control_vs_alzheimer.png", "positive": "AD",
    },
    {
        "label": "F4", "file": "oof_predictions_lateralized_v1_control_vs_frontotemporal_dementia_psd_plus_hcfim_logistic_elasticnet.csv",
        "output": "classification_best_lateralized_v1_control_vs_frontotemporal_dementia.png", "positive": "FTD",
    },
]

audit_rows = []
for spec in figure_specs:
    pred = pd.read_csv(RESULTS / spec["file"])
    y_true = pred["y_true"].to_numpy(int)
    y_prob = pred["y_prob"].to_numpy(float)
    y_pred = pred["y_pred"].to_numpy(int)
    auc_value = roc_auc_score(y_true, y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1]).astype(int)
    row_totals = cm.sum(axis=1)
    percentages = cm / row_totals[:, None] * 100
    annotations = np.array([
        [f"{cm[i, j]}/{row_totals[i]}\n({percentages[i, j]:.1f}%)" for j in range(2)]
        for i in range(2)
    ])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    axes[0].plot(fpr, tpr, color="#2F5597", linewidth=2.2, label=f"AUC = {auc_value:.3f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="#999999", linewidth=1)
    axes[0].set(xlabel="False-positive rate", ylabel="True-positive rate")
    axes[0].legend(loc="lower right")
    sns.heatmap(
        percentages, annot=annotations, fmt="", cmap="Blues", cbar=False,
        xticklabels=["control", "pathology"], yticklabels=["control", "pathology"], ax=axes[1],
    )
    axes[1].set(xlabel="Predicted", ylabel="Observed", title=f"Threshold = 0.50 | n = {len(pred)}")
    out_path = FIGURES / spec["output"]
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    audit_rows.append({"figure": spec["label"], "n": len(pred), "auc": auc_value, "TN": cm[0,0], "FP": cm[0,1], "FN": cm[1,0], "TP": cm[1,1]})

figure_audit = pd.DataFrame(audit_rows)
figure_audit.to_csv(REPAIR_RESULTS / "figures_f1_f4_audit.csv", index=False)
display(figure_audit)

### 3. Reconstruct Table H2

In [ ]:
embedding_dir = RESULTS / "embedding_dimension_selection_validation"
embedding_summary = pd.read_csv(embedding_dir / "repeated_nested_embedding_dimension_summary.csv")
selection_frequency = pd.read_csv(embedding_dir / "inner_fold_m_selection_frequency.csv")

frequency_text = {}
for task, group in selection_frequency.groupby("task"):
    frequency_text[task] = "; ".join(
        f"m={int(row.m)}: {row.selection_fraction:.0%}" for row in group.itertuples(index=False)
    )

table_h2 = embedding_summary[["task", "method", "auc_mean", "auc_p2_5", "auc_p97_5"]].copy()
table_h2["inner_selection_frequency"] = "n/a"
adaptive = table_h2["method"].eq("inner_selected_m")
table_h2.loc[adaptive, "inner_selection_frequency"] = table_h2.loc[adaptive, "task"].map(frequency_text)
table_h2["interpretation"] = ""
m3_ad = (table_h2["task"] == "control_vs_alzheimer") & (table_h2["method"] == "fixed_m3")
table_h2.loc[m3_ad, "interpretation"] = (
    "Predominantly below chance across repetitions; interpreted as unstable anti-learning, not useful discrimination."
)
assert table_h2.loc[~adaptive, "inner_selection_frequency"].eq("n/a").all()
assert table_h2.loc[adaptive, "inner_selection_frequency"].ne("n/a").all()
table_h2.to_csv(embedding_dir / "table_h2_corrected.csv", index=False)
display(table_h2.style.format({"auc_mean": "{:.3f}", "auc_p2_5": "{:.3f}", "auc_p97_5": "{:.3f}"}))

## Takeaways

The publication figures now distinguish the original exploratory partitions from the repeated matched-$k$ evidence. Figure 4 uses a controlled comparison, F1–F4 use integer counts, and Table H2 no longer assigns adaptive selection frequencies to fixed strategies. Notebook 07 must be executed after its $k=30$ correction to regenerate G2, G3, and G5.